In [12]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.Stacks import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPrismaticCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import CathodeMaterial, AnodeMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import PrismaticCase, PrismaticLid, PrismaticShell

import pandas as pd
import plotly.express as px

In [13]:
# construct cathode
cathode_active_material = CathodeMaterial(name="Faradion_Gen2_4.25V", 
                                            specific_cost=11.26, 
                                            density=4, 
                                            irreversible_capacity_scaling=1, 
                                            reversible_capacity_scaling=1)

cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)

cathode_binder = Binder(specific_cost=15, density=1.7)

cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: 89},
                                           binder={cathode_binder: 5},
                                           conductive_additive={cathode_conductive_additive: 6})

cathode_current_collector = CurrentCollector(formula="Cu", 
                                             thickness=5, 
                                             length=12.5,
                                             width=10.8,
                                             bare_tab_area=8.22)

cathode = Cathode(formulation=cathode_formulation,
                    mass_loading=10.68,
                    current_collector=cathode_current_collector,
                    calender_density=2.60)


In [14]:
# construct anode
anode_active_material = AnodeMaterial(name="Faradion_HC",
                                      specific_cost=14.27,
                                      density=1.50,
                                      irreversible_capacity_scaling=1,
                                      reversible_capacity_scaling=1)

anode_conductive_additive = ConductiveAdditive(specific_cost=9, 
                                               density=1.9)

anode_binder = Binder(specific_cost=10, 
                      density=1.7)

anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: 88},
                                         binder={anode_binder: 3},
                                         conductive_additive={anode_conductive_additive: 9})

anode_current_collector = CurrentCollector(formula="Cu",
                                            thickness=5,
                                            length=12.5,
                                            width=10.8,
                                            bare_tab_area=7.55)

anode = Anode(formulation=anode_formulation,
              mass_loading=5.25,
              current_collector=anode_current_collector,
              calender_density=0.85)


In [15]:
# construct seperator
separator = Separator(thickness=16, 
                      areal_cost=0.9, 
                      density=0.4, 
                      slit_width=11.0, 
                      porosity=47, 
                      fold_length=12.8)

In [16]:

# make the case
prismatic_shell = PrismaticShell(cost=0.15,
                                 mass=60,
                                 internal_length=13.0,
                                 internal_width=11.5,
                                 internal_height=1.8,
                                 wall_thickness=0.8)

prismatic_lid = PrismaticLid(cost=0.05,
                             mass=10,
                             external_width=1.3, 
                             internal_width=0.8)

prismatic_case = PrismaticCase(lid=prismatic_lid, shell=prismatic_shell)

# construct the stack
stack = prismatic_case.get_optimized_stack(anode=anode,
                                           cathode=cathode,
                                           separator=separator)


In [17]:
# make electrolyte
electrolyte = Electrolyte(specific_cost=8.94, 
                          density=1.2)

cell = StackedPrismaticCell(prismatic_case=prismatic_case,
                            stack=stack,
                            electrolyte=electrolyte,
                            electrolyte_overfill=10,
                            reversible_capacity=30.000,
                            irreversible_capacity=0.400,
                            grid_n=200)

In [18]:
figure = cell.get_capacity_voltage_plot(width=900, height=600)
figure.show()

In [19]:
print(f"The number of stacks is {cell.stack.n_stacks}")
print(f"The energy the cell can hold is {cell.energy} Wh")
print(f"The specific energy of the cell is {cell.specific_energy} Wh/kg")
print(f"The cost of the cell is {cell.cost} $")
print(f"The normalized cost of the cell is {cell.normalized_cost} $/kWh")
print(f"The energy density of the cell is {cell.energy_density} Wh/L")

The number of stacks is 71
The energy the cell can hold is 76.68 Wh
The specific energy of the cell is 127.77 Wh/kg
The cost of the cell is 7.76 $
The normalized cost of the cell is 101.26 $/kWh
The energy density of the cell is 229.37 Wh/L


In [20]:
cell.mass_breakdown

{Stack: np.float64(411.553), Electrolyte: 118.533, Prismatic Case: 70.0}

In [21]:
figure = cell.get_cost_breakdown_plot(width=1200, height=500)
figure.show() 

In [22]:
figure = cell.get_mass_breakdown_plot(width=1200, height=500)
figure.show()